# ML Project / NASDAQ Prediction and Analysis (big dataset)

Same pipeline as `1_Project_Features.ipynb`, but starting in 2005 instead of 2022. The NQ 1-minute file (Kaggle) only covers Dec 2022 - Dec 2025, so it is replaced by the Asian indices (Nikkei 225, Hang Seng), which close before our 09:00 ET cutoff and go back decades on yfinance. Bitcoin is dropped (it only starts in 2014).

**Objective** : Predicting if the Nasdaq-100 (`^NDX`) goes up or down **during the New York session of day `t`** (from the open at 09:30 ET to the close at 16:00 ET), using only information known **before 09:00 ET of day `t`**. The overnight information is how Asia (Tokyo, Hong Kong) traded on day `t`; the daily data (stocks, macro) is used as context from the day before.


In [114]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import requests 
import io
import os
from sklearn.model_selection import TimeSeriesSplit

In [115]:
path_to_data = ("./data")

## Data

We take the current Nasdaq-100 list from Wikipedia. This means we use today's companies for the whole period, so there's some survivorship bias (we'll mention it in the report).

With 20 years of history this bias is bigger: only the companies that survived until today are in the list.

In [116]:
html = requests.get("https://en.wikipedia.org/wiki/List_of_NASDAQ-100_companies", headers={"User-Agent": "Mozilla/5.0"}).text
table = pd.read_html(io.StringIO(html))
ndx = next(t for t in table if "Ticker" in t.columns)
TICKERS = ndx["Ticker"].str.replace(".", "-", regex=False).tolist()

In [117]:
TICKERS

['ADBE',
 'AMD',
 'ABNB',
 'ALNY',
 'GOOGL',
 'GOOG',
 'AMZN',
 'AEP',
 'AMGN',
 'ADI',
 'AAPL',
 'AMAT',
 'APP',
 'ARM',
 'ASML',
 'ALAB',
 'ADSK',
 'ADP',
 'AXON',
 'BKR',
 'BKNG',
 'AVGO',
 'CDNS',
 'CTAS',
 'CSCO',
 'CCEP',
 'CMCSA',
 'CEG',
 'CPRT',
 'CRWV',
 'COST',
 'CRWD',
 'CSX',
 'DDOG',
 'DXCM',
 'FANG',
 'DASH',
 'EXC',
 'FAST',
 'FER',
 'FTNT',
 'GEHC',
 'GILD',
 'HONA',
 'HON',
 'IDXX',
 'INTC',
 'INTU',
 'ISRG',
 'KDP',
 'KLAC',
 'LRCX',
 'LIN',
 'LITE',
 'MAR',
 'MRVL',
 'MELI',
 'META',
 'MCHP',
 'MU',
 'MSFT',
 'MSTR',
 'MRNA',
 'MDLZ',
 'MPWR',
 'MNST',
 'NBIS',
 'NFLX',
 'NVDA',
 'NXPI',
 'ORLY',
 'ODFL',
 'PCAR',
 'PLTR',
 'PANW',
 'PAYX',
 'PYPL',
 'PDD',
 'PEP',
 'QCOM',
 'REGN',
 'RKLB',
 'ROP',
 'ROST',
 'SNDK',
 'STX',
 'SHOP',
 'SPCX',
 'SBUX',
 'SNPS',
 'TMUS',
 'TTWO',
 'TER',
 'TSLA',
 'TXN',
 'TRI',
 'VRTX',
 'WMT',
 'WDC',
 'WDAY',
 'XEL']

On top of the stocks we add some macro data: volatility indices (VIX, VXN), treasury yields, oil, gold, silver, the dollar and Bitcoin. The idea is to give the model some context about the market, not just prices.

In [118]:
NASDAQ_FEATURES = {
    # --- INDICES ET VOLATILITÉ ---                    
    "^IXIC": "Nasdaq_composite",              # Broad tech market index
    "^VXN": "Nasdaq_volatility",              # Nasdaq-100 implied volatility (Tech fear index)
    "^VIX": "Volatility",                     # S&P 500 implied volatility (Market fear index)

    # --- interest ---
    "^IRX": "3month_treasury_yield",          # 3-month Treasury yield
    "^FVX": "5y_treasury_yield",              # 5-year Treasury yield
    "^TNX": "10y_treasury_yield",             # 10-year Treasury yield (Tech discounting rate)
    "^TYX": "30y_treasury_yield",             # 30-year Treasury yield

    # --- MATIÈRES PREMIÈRES ET DEVISES ---
    "DX-Y.NYB": "US_dollar",                  # US dollar index (DXY)
    "CL=F": "crude_oil",                      # WTI crude oil futures
    "GC=F": "gold",                           # Gold futures
    "SI=F": "silver",                         # Silver futures
    # ponytail: BTC-USD removed, it starts in 2014 and the final dropna would cut the data back to 2014
}
tickers1 = list(NASDAQ_FEATURES.keys())

Asian indices. Tokyo closes at 15:30 JST (01:30 / 02:30 ET) and Hong Kong at 16:00 HKT (03:00 / 04:00 ET), so the close of day `t` is known before the 09:00 ET cutoff. Europe closes at 11:30 ET, after the cutoff, so we don't use it.

In [119]:
ASIA = {
    "^N225": "Nikkei_225",
    "^HSI": "Hang_Seng",
}
tickers2 = list(ASIA.keys())

In [120]:
TICKERS = list(dict.fromkeys(TICKERS + tickers1 + tickers2))

In [121]:
df_nasdaq = yf.download("^NDX", start="2018-01-01", end="2025-12-12",
                     auto_adjust=True, progress=False)[['Open', 'Close','Volume']]
print(df_nasdaq.columns)
print(df_nasdaq.tail())

MultiIndex([(  'Open', '^NDX'),
            ( 'Close', '^NDX'),
            ('Volume', '^NDX')],
           names=['Price', 'Ticker'])
Price               Open         Close      Volume
Ticker              ^NDX          ^NDX        ^NDX
Date                                              
2025-12-05  25657.779297  25692.050781  8248280000
2025-12-08  25758.820312  25627.949219  8533670000
2025-12-09  25593.669922  25668.689453  7192600000
2025-12-10  25631.589844  25776.439453  7987900000
2025-12-11  25598.390625  25686.689453  8337770000


In [122]:
df_nasdaq.head(20)

Price,Open,Close,Volume
Ticker,^NDX,^NDX,^NDX
Date,,,
2018-01-02,6431.589844,6511.339844,1929700000
2018-01-03,6520.029785,6575.799805,2173130000
2018-01-04,6595.750000,6584.580078,2103220000
2018-01-05,6613.129883,6653.290039,2024000000
2018-01-08,6651.680176,6676.629883,2055010000
2018-01-09,6692.870117,6677.939941,2155810000
2018-01-10,6640.660156,6662.660156,2132430000
2018-01-11,6675.689941,6708.490234,2015170000


In [123]:
raw = yf.download(TICKERS, start="2018-01-01", end="2025-12-12")[['Open', 'Close','Volume']]

[******                12%                       ]  14 of 114 completed$HONA: Data doesn't exist for startDate = 1514782800, endDate = 1765515600
[**********************85%****************       ]  97 of 114 completed$SPCX: Data doesn't exist for startDate = 1514782800, endDate = 1765515600
[**********************98%********************** ]  112 of 114 completed$KLAC: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-12-12)
[*********************100%***********************]  114 of 114 completed

3 Failed downloads:
['HONA', 'SPCX']: Data doesn't exist for startDate = 1514782800, endDate = 1765515600
['KLAC']: possibly delisted; no price data found  (1d 2018-01-01 -> 2025-12-12)


In [124]:
raw.head(10)

Price            Open                                                     \
Ticker           AAPL ABNB        ADBE        ADI        ADP        ADSK   
Date                                                                       
2018-01-02  39.711742  NaN  175.850006  76.266428  95.828744  105.339996   
2018-01-03  40.264839  NaN  178.000000  77.155233  95.952641  107.000000   
2018-01-04  40.267193  NaN  181.929993  78.514081  97.150215  110.129997   
2018-01-05  40.477227  NaN  185.000000  78.214960  97.910003  113.070000   
2018-01-08  40.689598  NaN  184.949997  78.462793  97.761370  110.419998   
2018-01-09  40.736273  NaN  185.570007  78.454255  97.241033  111.570000   
2018-01-10  40.411875  NaN  185.210007  77.838910  97.687015  111.709999   
2018-01-11  40.745611  NaN  187.289993  77.257801  97.257571  111.669998   
2018-01-12  41.116676  NaN  189.089996  77.821829  97.298854  113.570000   
2018-01-15        NaN  NaN         NaN        NaN        NaN         NaN   

Price                                              ...     Volume       \
Ticker            AEP ALAB        ALNY       AMAT  ...        XEL ^FVX   
Date                                               ...                   
2018-01-02  53.465115  NaN  128.320007  46.954335  ...  2443400.0  0.0   
2018-01-03  52.756436  NaN  131.380005  48.644257  ...  4114900.0  0.0   
2018-01-04  52.369230  NaN  134.940002  49.516473  ...  2807000.0  0.0   
2018-01-05  51.960098  NaN  131.960007  49.652765  ...  3728000.0  0.0   
2018-01-08  51.748207  NaN  130.330002  49.979840  ...  3837600.0  0.0   
2018-01-09  52.091602  NaN  129.800003  51.033770  ...  3006300.0  0.0   
2018-01-10  51.368310  NaN  129.000000  49.143975  ...  2690000.0  0.0   
2018-01-11  50.842267  NaN  129.320007  48.580658  ...  3020900.0  0.0   
2018-01-12  50.024000  NaN  127.949997  48.371692  ...  3422600.0  0.0   
2018-01-15        NaN  NaN         NaN        NaN  ...        NaN  NaN   

Price                                                                         
Ticker              ^HSI ^IRX         ^IXIC        ^N225 ^TNX ^TYX ^VIX ^VXN  
Date                                                                          
2018-01-02  2.442300e+09  0.0  1.929700e+09          NaN  0.0  0.0  0.0  0.0  
2018-01-03  2.309500e+09  0.0  2.173130e+09          NaN  0.0  0.0  0.0  0.0  
2018-01-04  2.995600e+09  0.0  2.103220e+09  102200000.0  0.0  0.0  0.0  0.0  
2018-01-05  2.263900e+09  0.0  2.024000e+09  101900000.0  0.0  0.0  0.0  0.0  
2018-01-08  2.004200e+09  0.0  2.055010e+09          NaN  0.0  0.0  0.0  0.0  
2018-01-09  2.004900e+09  0.0  2.155810e+09   94100000.0  0.0  0.0  0.0  0.0  
2018-01-10  2.595400e+09  0.0  2.132430e+09   88800000.0  0.0  0.0  0.0  0.0  
2018-01-11  1.794800e+09  0.0  2.015170e+09   83700000.0  0.0  0.0  0.0  0.0  
2018-01-12  2.327400e+09  0.0  1.986600e+09   93700000.0  0.0  0.0  0.0  0.0  
2018-01-15  2.664100e+09  NaN           NaN   75400000.0  NaN  NaN  NaN  NaN  

[10 rows x 342 columns]

We join everything on the Nasdaq-100 dates (inner join). Bitcoin trades on weekends but the index doesn't, so this way we only keep real trading days and we don't need to fill anything.

In [125]:
raw = pd.concat((df_nasdaq,raw),axis=1,join='inner',)

In [126]:
raw

Price               Open         Close      Volume        Open              \
Ticker              ^NDX          ^NDX        ^NDX        AAPL        ABNB   
Date                                                                         
2018-01-02   6431.589844   6511.339844  1929700000   39.711742         NaN   
2018-01-03   6520.029785   6575.799805  2173130000   40.264839         NaN   
2018-01-04   6595.750000   6584.580078  2103220000   40.267193         NaN   
2018-01-05   6613.129883   6653.290039  2024000000   40.477227         NaN   
2018-01-08   6651.680176   6676.629883  2055010000   40.689598         NaN   
...                  ...           ...         ...         ...         ...   
2025-12-05  25657.779297  25692.050781  8248280000  279.566064  120.919998   
2025-12-08  25758.820312  25627.949219  8533670000  277.164420  124.214996   
2025-12-09  25593.669922  25668.689453  7192600000  277.194290  121.610001   
2025-12-10  25631.589844  25776.439453  7987900000  276.785741  125.449997   
2025-12-11  25598.390625  25686.689453  8337770000  278.131020  127.599998   

Price                                                                   ...  \
Ticker            ADBE         ADI         ADP        ADSK         AEP  ...   
Date                                                                    ...   
2018-01-02  175.850006   76.266428   95.828744  105.339996   53.465115  ...   
2018-01-03  178.000000   77.155233   95.952641  107.000000   52.756436  ...   
2018-01-04  181.929993   78.514081   97.150215  110.129997   52.369230  ...   
2018-01-05  185.000000   78.214960   97.910003  113.070000   51.960098  ...   
2018-01-08  184.949997   78.462793   97.761370  110.419998   51.748207  ...   
...                ...         ...         ...         ...         ...  ...   
2025-12-05  330.600006  276.659808  252.083831  306.000000  114.353964  ...   
2025-12-08  347.450012  279.282235  253.502640  307.600006  114.033799  ...   
2025-12-09  341.399994  275.119470  251.782584  301.070007  112.811378  ...   
2025-12-10  342.130005  273.206560  250.101377  299.459991  113.180056  ...   
2025-12-11  341.649994  279.074116  252.268475  299.929993  111.094189  ...   

Price          Volume                                                          \
Ticker            XEL ^FVX          ^HSI ^IRX         ^IXIC        ^N225 ^TNX   
Date                                                                            
2018-01-02  2443400.0  0.0  2.442300e+09  0.0  1.929700e+09          NaN  0.0   
2018-01-03  4114900.0  0.0  2.309500e+09  0.0  2.173130e+09          NaN  0.0   
2018-01-04  2807000.0  0.0  2.995600e+09  0.0  2.103220e+09  102200000.0  0.0   
2018-01-05  3728000.0  0.0  2.263900e+09  0.0  2.024000e+09  101900000.0  0.0   
2018-01-08  3837600.0  0.0  2.004200e+09  0.0  2.055010e+09          NaN  0.0   
...               ...  ...           ...  ...           ...          ...  ...   
2025-12-05  3563000.0  0.0  2.973100e+09  0.0  8.248280e+09  119300000.0  0.0   
2025-12-08  3197700.0  0.0  2.923100e+09  0.0  8.533670e+09  104400000.0  0.0   
2025-12-09  4842400.0  0.0  2.858700e+09  0.0  7.192600e+09  107100000.0  0.0   
2025-12-10  4824700.0  0.0  2.729900e+09  0.0  7.987900e+09  125100000.0  0.0   
2025-12-11  5494900.0  0.0  2.352600e+09  0.0  8.337770e+09  109900000.0  0.0   

Price                      
Ticker     ^TYX ^VIX ^VXN  
Date                       
2018-01-02  0.0  0.0  0.0  
2018-01-03  0.0  0.0  0.0  
2018-01-04  0.0  0.0  0.0  
2018-01-05  0.0  0.0  0.0  
2018-01-08  0.0  0.0  0.0  
...         ...  ...  ...  
2025-12-05  0.0  0.0  0.0  
2025-12-08  0.0  0.0  0.0  
2025-12-09  0.0  0.0  0.0  
2025-12-10  0.0  0.0  0.0  
2025-12-11  0.0  0.0  0.0  

[1998 rows x 345 columns]

### Missing values

Many companies are recent compared to 2005 (ARM, ABNB, PLTR...), so they have no history for most of the period. We drop the columns with less than 40% of data. Later the stock returns only keep the stocks with the full history.

We also drop GOOG (almost the same as GOOGL) and ^IXIC (it moves with the Nasdaq-100), they don't add anything new.

In [127]:
dropping = raw.isnull().sum().sort_values(ascending=False)/len(raw)

In [128]:
dropping

Price   Ticker
Open    SPCX      1.0
Volume  HONA      1.0
        KLAC      1.0
Open    KLAC      1.0
        HONA      1.0
                 ... 
Close   AMAT      0.0
        ALNY      0.0
        AEP       0.0
        ADSK      0.0
Volume  ^VXN      0.0
Length: 345, dtype: float64

In [129]:
raw_drop = raw.dropna(axis=1, thresh=int(0.4 * len(raw)))
# GOOG is basically GOOGL and ^IXIC moves with ^NDX, so they add nothing
raw_drop = raw_drop.drop(columns=["GOOG", "^IXIC"], level=1)

In [130]:
print(list(set(raw.columns.tolist()) - set(raw_drop.columns.tolist())))

[('Volume', 'SNDK'), ('Open', 'CRWV'), ('Volume', 'SPCX'), ('Volume', 'KLAC'), ('Volume', 'HONA'), ('Close', 'CRWV'), ('Volume', 'ALAB'), ('Open', 'SNDK'), ('Open', 'SPCX'), ('Open', 'KLAC'), ('Close', 'SNDK'), ('Open', 'HONA'), ('Volume', 'GEHC'), ('Open', 'ALAB'), ('Close', 'SPCX'), ('Close', 'KLAC'), ('Volume', 'NBIS'), ('Close', 'HONA'), ('Close', 'ALAB'), ('Open', 'GEHC'), ('Open', 'NBIS'), ('Close', 'GEHC'), ('Volume', 'ARM'), ('Close', 'NBIS'), ('Volume', '^IXIC'), ('Open', 'ARM'), ('Volume', 'GOOG'), ('Close', 'ARM'), ('Open', '^IXIC'), ('Open', 'GOOG'), ('Close', '^IXIC'), ('Volume', 'CRWV'), ('Close', 'GOOG')]


In [131]:
try:
    raw_drop.loc[:, (slice(None), "SPCX")]
except KeyError:
    print("SPCX not in raw_drop: it was dropped")

SPCX not in raw_drop: it was dropped


In [132]:
raw.loc[:, (slice(None), "SPCX")]

Price,Open,Close,Volume
Ticker,SPCX,SPCX,SPCX
Date,,,
2018-01-02,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN
...,...,...,...
2025-12-05,NaN,NaN,NaN
2025-12-08,NaN,NaN,NaN


In [133]:
raw_drop.describe()

Price           Open         Close        Volume         Open               \
Ticker          ^NDX          ^NDX          ^NDX         AAPL         ABNB   
count    1998.000000   1998.000000  1.998000e+03  1998.000000  1257.000000   
mean    13410.646030  13413.968320  4.748527e+09   134.017627   137.686500   
std      5137.796436   5137.151191  2.240480e+09    65.792159    24.297623   
min      5969.080078   5899.350098  9.589500e+08    34.106345    82.970001   
25%      8287.307617   8299.497803  2.689580e+09    63.843057   120.489998   
50%     12969.640137  12968.479980  4.582590e+09   141.927574   134.809998   
75%     16513.059570  16528.814941  5.588708e+09   181.258845   151.339996   
max     26147.720703  26119.849609  1.630873e+10   285.206384   216.240005   

Price                                                                    ...  \
Ticker         ADBE          ADI          ADP         ADSK          AEP  ...   
count   1998.000000  1998.000000  1998.000000  1998.000000  1998.000000  ...   
mean     407.269740   146.973723   187.827870   220.611982    74.862658  ...   
std      121.034193    49.852612    58.875514    58.980768    14.828727  ...   
min      175.850006    67.266355    90.328303   105.339996    46.721028  ...   
25%      302.214996   100.115483   135.708507   173.000000    66.268547  ...   
50%      396.184998   149.025874   190.178473   215.570000    72.514881  ...   
75%      499.425003   181.724756   230.135919   272.182503    82.177714  ...   
max      696.280029   279.282235   316.343099   342.519989   120.708565  ...   

Price         Volume                                              \
Ticker           WMT           XEL    ^FVX          ^HSI    ^IRX   
count   1.998000e+03  1.998000e+03  1998.0  1.903000e+03  1998.0   
mean    2.217852e+07  3.477599e+06     0.0  2.431984e+09     0.0   
std     1.276604e+07  1.723826e+06     0.0  1.033757e+09     0.0   
min     6.287500e+06  0.000000e+00     0.0  4.840000e+08     0.0   
25%     1.483545e+07  2.443475e+06     0.0  1.745450e+09     0.0   
50%     1.896375e+07  3.086150e+06     0.0  2.203800e+09     0.0   
75%     2.524935e+07  4.001800e+06     0.0  2.860500e+09     0.0   
max     1.562655e+08  2.278220e+07     0.0  1.268650e+10     0.0   

Price                                                 
Ticker         ^N225    ^TNX    ^TYX    ^VIX    ^VXN  
count   1.871000e+03  1998.0  1998.0  1998.0  1998.0  
mean    8.870102e+07     0.0     0.0     0.0     0.0  
std     3.299714e+07     0.0     0.0     0.0     0.0  
min     0.000000e+00     0.0     0.0     0.0     0.0  
25%     6.525000e+07     0.0     0.0     0.0     0.0  
50%     7.970000e+07     0.0     0.0     0.0     0.0  
75%     1.070500e+08     0.0     0.0     0.0     0.0  
max     2.856000e+08     0.0     0.0     0.0     0.0  

[8 rows x 312 columns]

From the date we keep only `Dayofweek`. The other date columns (year, month, elapsed...) are trends or noise for this target.

In [134]:
raw_drop = raw_drop.reset_index()
raw_time = raw_drop.copy()
# ponytail: only Dayofweek was kept from fastai add_datepart, so no fastai needed
raw_time[("Dayofweek", "")] = raw_time[("Date", "")].dt.dayofweek

/var/folders/xl/0wxp8zgx2wz1_sw2nyhksnr00000gn/T/ipykernel_40208/2172066189.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  raw_drop = raw_drop.reset_index()


In [135]:
raw_time.isnull().sum().sort_values(ascending=False)/len(raw)

Price      Ticker
Open       CEG       0.510010
Volume     CEG       0.510010
Close      CEG       0.510010
Open       APP       0.413413
Close      APP       0.413413
                       ...   
           ADP       0.000000
           ADI       0.000000
           ADBE      0.000000
           AAPL      0.000000
Dayofweek            0.000000
Length: 314, dtype: float64

In [136]:
columns = raw_time.columns.to_list()
print(columns)

[('Date', ''), ('Open', '^NDX'), ('Close', '^NDX'), ('Volume', '^NDX'), ('Open', 'AAPL'), ('Open', 'ABNB'), ('Open', 'ADBE'), ('Open', 'ADI'), ('Open', 'ADP'), ('Open', 'ADSK'), ('Open', 'AEP'), ('Open', 'ALNY'), ('Open', 'AMAT'), ('Open', 'AMD'), ('Open', 'AMGN'), ('Open', 'AMZN'), ('Open', 'APP'), ('Open', 'ASML'), ('Open', 'AVGO'), ('Open', 'AXON'), ('Open', 'BKNG'), ('Open', 'BKR'), ('Open', 'CCEP'), ('Open', 'CDNS'), ('Open', 'CEG'), ('Open', 'CL=F'), ('Open', 'CMCSA'), ('Open', 'COST'), ('Open', 'CPRT'), ('Open', 'CRWD'), ('Open', 'CSCO'), ('Open', 'CSX'), ('Open', 'CTAS'), ('Open', 'DASH'), ('Open', 'DDOG'), ('Open', 'DX-Y.NYB'), ('Open', 'DXCM'), ('Open', 'EXC'), ('Open', 'FANG'), ('Open', 'FAST'), ('Open', 'FER'), ('Open', 'FTNT'), ('Open', 'GC=F'), ('Open', 'GILD'), ('Open', 'GOOGL'), ('Open', 'HON'), ('Open', 'IDXX'), ('Open', 'INTC'), ('Open', 'INTU'), ('Open', 'ISRG'), ('Open', 'KDP'), ('Open', 'LIN'), ('Open', 'LITE'), ('Open', 'LRCX'), ('Open', 'MAR'), ('Open', 'MCHP'), 

In [137]:
raw_time = raw_time.copy(True)

In [138]:
raw_time.loc[:, (slice(None),'^NDX')]

Price,Open,Close,Volume
Ticker,^NDX,^NDX,^NDX
0,6431.589844,6511.339844,1929700000
1,6520.029785,6575.799805,2173130000
2,6595.750000,6584.580078,2103220000
3,6613.129883,6653.290039,2024000000
4,6651.680176,6676.629883,2055010000
...,...,...,...
1993,25657.779297,25692.050781,8248280000
1994,25758.820312,25627.949219,8533670000
1995,25593.669922,25668.689453,7192600000


## Target

We decide at 09:00 ET of day `t`, we enter at the NY open of day `t` (09:30) and we exit at the NY close of the same day (16:00).

The target is the open-to-close log return of `^NDX`, and the label is 1 if it is positive, 0 if not. It is open-to-close on purpose: the overnight move is already in the open price. The `Open` and `Close` of `^NDX` of day `t` are only used here, never as features.

In [139]:
ndx = raw_drop.set_index('Date')
close = ndx['Close']['^NDX']
open_ = ndx['Open']['^NDX']

In [140]:
open_

Date
2018-01-02     6431.589844
2018-01-03     6520.029785
2018-01-04     6595.750000
2018-01-05     6613.129883
2018-01-08     6651.680176
                  ...     
2025-12-05    25657.779297
2025-12-08    25758.820312
2025-12-09    25593.669922
2025-12-10    25631.589844
2025-12-11    25598.390625
Name: ^NDX, Length: 1998, dtype: float64

In [141]:
ret_ny = np.log(close / open_)

In [142]:
ret_ny

Date
2018-01-02    0.012323
2018-01-03    0.008517
2018-01-04   -0.001695
2018-01-05    0.006054
2018-01-08    0.003744
                ...   
2025-12-05    0.001335
2025-12-08   -0.005094
2025-12-09    0.002927
2025-12-10    0.005635
2025-12-11    0.003443
Name: ^NDX, Length: 1998, dtype: float64

In [143]:
raw_time = raw_time.set_index("Date")

In [144]:
raw_time

Price               Open         Close      Volume        Open              \
Ticker              ^NDX          ^NDX        ^NDX        AAPL        ABNB   
Date                                                                         
2018-01-02   6431.589844   6511.339844  1929700000   39.711742         NaN   
2018-01-03   6520.029785   6575.799805  2173130000   40.264839         NaN   
2018-01-04   6595.750000   6584.580078  2103220000   40.267193         NaN   
2018-01-05   6613.129883   6653.290039  2024000000   40.477227         NaN   
2018-01-08   6651.680176   6676.629883  2055010000   40.689598         NaN   
...                  ...           ...         ...         ...         ...   
2025-12-05  25657.779297  25692.050781  8248280000  279.566064  120.919998   
2025-12-08  25758.820312  25627.949219  8533670000  277.164420  124.214996   
2025-12-09  25593.669922  25668.689453  7192600000  277.194290  121.610001   
2025-12-10  25631.589844  25776.439453  7987900000  276.785741  125.449997   
2025-12-11  25598.390625  25686.689453  8337770000  278.131020  127.599998   

Price                                                                   ...  \
Ticker            ADBE         ADI         ADP        ADSK         AEP  ...   
Date                                                                    ...   
2018-01-02  175.850006   76.266428   95.828744  105.339996   53.465115  ...   
2018-01-03  178.000000   77.155233   95.952641  107.000000   52.756436  ...   
2018-01-04  181.929993   78.514081   97.150215  110.129997   52.369230  ...   
2018-01-05  185.000000   78.214960   97.910003  113.070000   51.960098  ...   
2018-01-08  184.949997   78.462793   97.761370  110.419998   51.748207  ...   
...                ...         ...         ...         ...         ...  ...   
2025-12-05  330.600006  276.659808  252.083831  306.000000  114.353964  ...   
2025-12-08  347.450012  279.282235  253.502640  307.600006  114.033799  ...   
2025-12-09  341.399994  275.119470  251.782584  301.070007  112.811378  ...   
2025-12-10  342.130005  273.206560  250.101377  299.459991  113.180056  ...   
2025-12-11  341.649994  279.074116  252.268475  299.929993  111.094189  ...   

Price          Volume                                                      \
Ticker            XEL ^FVX          ^HSI ^IRX        ^N225 ^TNX ^TYX ^VIX   
Date                                                                        
2018-01-02  2443400.0  0.0  2.442300e+09  0.0          NaN  0.0  0.0  0.0   
2018-01-03  4114900.0  0.0  2.309500e+09  0.0          NaN  0.0  0.0  0.0   
2018-01-04  2807000.0  0.0  2.995600e+09  0.0  102200000.0  0.0  0.0  0.0   
2018-01-05  3728000.0  0.0  2.263900e+09  0.0  101900000.0  0.0  0.0  0.0   
2018-01-08  3837600.0  0.0  2.004200e+09  0.0          NaN  0.0  0.0  0.0   
...               ...  ...           ...  ...          ...  ...  ...  ...   
2025-12-05  3563000.0  0.0  2.973100e+09  0.0  119300000.0  0.0  0.0  0.0   
2025-12-08  3197700.0  0.0  2.923100e+09  0.0  104400000.0  0.0  0.0  0.0   
2025-12-09  4842400.0  0.0  2.858700e+09  0.0  107100000.0  0.0  0.0  0.0   
2025-12-10  4824700.0  0.0  2.729900e+09  0.0  125100000.0  0.0  0.0  0.0   
2025-12-11  5494900.0  0.0  2.352600e+09  0.0  109900000.0  0.0  0.0  0.0   

Price           Dayofweek  
Ticker     ^VXN            
Date                       
2018-01-02  0.0         1  
2018-01-03  0.0         2  
2018-01-04  0.0         3  
2018-01-05  0.0         4  
2018-01-08  0.0         0  
...         ...       ...  
2025-12-05  0.0         4  
2025-12-08  0.0         0  
2025-12-09  0.0         1  
2025-12-10  0.0         2  
2025-12-11  0.0         3  

[1998 rows x 313 columns]

In [145]:
data = raw_time.copy()
# both have the Date index, so pandas aligns them by date
data["ret_ny"] = ret_ny

In [146]:
r = data["ret_ny"]
data["y_ny"] = (r > 0).astype(float).where(r.notna())

In [147]:
print(data[["ret_ny", "y_ny"]].tail(7))

Price         ret_ny y_ny
Ticker                   
Date                     
2025-12-03  0.006379  1.0
2025-12-04 -0.002984  0.0
2025-12-05  0.001335  1.0
2025-12-08 -0.005094  0.0
2025-12-09  0.002927  1.0
2025-12-10  0.005635  1.0
2025-12-11  0.003443  1.0


In [148]:
print(data.columns.tolist())
print(data.index.min(), data.index.max(), len(data))

[('Open', '^NDX'), ('Close', '^NDX'), ('Volume', '^NDX'), ('Open', 'AAPL'), ('Open', 'ABNB'), ('Open', 'ADBE'), ('Open', 'ADI'), ('Open', 'ADP'), ('Open', 'ADSK'), ('Open', 'AEP'), ('Open', 'ALNY'), ('Open', 'AMAT'), ('Open', 'AMD'), ('Open', 'AMGN'), ('Open', 'AMZN'), ('Open', 'APP'), ('Open', 'ASML'), ('Open', 'AVGO'), ('Open', 'AXON'), ('Open', 'BKNG'), ('Open', 'BKR'), ('Open', 'CCEP'), ('Open', 'CDNS'), ('Open', 'CEG'), ('Open', 'CL=F'), ('Open', 'CMCSA'), ('Open', 'COST'), ('Open', 'CPRT'), ('Open', 'CRWD'), ('Open', 'CSCO'), ('Open', 'CSX'), ('Open', 'CTAS'), ('Open', 'DASH'), ('Open', 'DDOG'), ('Open', 'DX-Y.NYB'), ('Open', 'DXCM'), ('Open', 'EXC'), ('Open', 'FANG'), ('Open', 'FAST'), ('Open', 'FER'), ('Open', 'FTNT'), ('Open', 'GC=F'), ('Open', 'GILD'), ('Open', 'GOOGL'), ('Open', 'HON'), ('Open', 'IDXX'), ('Open', 'INTC'), ('Open', 'INTU'), ('Open', 'ISRG'), ('Open', 'KDP'), ('Open', 'LIN'), ('Open', 'LITE'), ('Open', 'LRCX'), ('Open', 'MAR'), ('Open', 'MCHP'), ('Open', 'MDLZ

The target doesn't look ahead anymore, so the last rows keep their target. We only remove the rows without target (if any).

In [149]:
target_cols = [("ret_ny", ""), ("y_ny", "")]

data = data.dropna(subset=[("y_ny", "")])

## Daily features

Context from the day before. All the daily US data closes at 16:00 or later, so everything here is shifted by 1 day (`shift(1)`): for day `t` we only use values up to the close of `t-1`.

Nasdaq-100: momentum over 1, 5 and 20 days, the NY session return of the day before, and the 20-day volatility of the daily returns.

In [150]:
ndx_close = data[("Close", "^NDX")]
for n in (1, 5, 20):
    data[(f"ndx_ret_{n}d", "")] = np.log(ndx_close / ndx_close.shift(n)).shift(1)

In [151]:
data[("ndx_ret_ny_prev", "")] = data[("ret_ny", "")].shift(1)
data[("ndx_vol_20d", "")] = np.log(ndx_close).diff().rolling(20).std().shift(1)

Macro: level for the volatility indices and the yields, daily log return for the dollar, oil and gold. Some of them don't trade on a few Nasdaq days (bond holidays), so we forward fill first (it only uses the past). Silver and the 5y / 30y yields are not used.

In [152]:
macro_close = data["Close"][["^VIX", "^VXN", "^TNX", "^IRX", "DX-Y.NYB", "CL=F", "GC=F"]].ffill()

In [153]:
data[("vix", "")] = macro_close["^VIX"].shift(1)
data[("vxn", "")] = macro_close["^VXN"].shift(1)
data[("yield_10y", "")] = macro_close["^TNX"].shift(1)
data[("yield_3m", "")] = macro_close["^IRX"].shift(1)

In [154]:
data[("dollar_ret", "")] = np.log(macro_close["DX-Y.NYB"]).diff().shift(1)
data[("oil_ret", "")] = np.log(macro_close["CL=F"]).diff().shift(1)
data[("gold_ret", "")] = np.log(macro_close["GC=F"]).diff().shift(1)

/opt/anaconda3/envs/NASDAQ_Prediction-Project/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


### Asia (day `t`)

Log return of the Nikkei and the Hang Seng from their previous close to their close of day `t`. **No shift**: these markets close before 09:00 ET of day `t`. If Asia is closed on day `t` (holiday) we forward fill, so the return is 0.

In [155]:
asia_close = data["Close"][tickers2].ffill()
data[("nikkei_ret", "")] = np.log(asia_close["^N225"]).diff()
data[("hsi_ret", "")] = np.log(asia_close["^HSI"]).diff()
asia_cols = [("nikkei_ret", ""), ("hsi_ret", "")]

In [156]:
# anti-leakage: Tokyo and Hong Kong close of day t is before 09:00 ET of day t (in any season)
assert (pd.Timestamp("2025-01-15 15:30", tz="Asia/Tokyo").tz_convert("America/New_York").hour < 9)
assert (pd.Timestamp("2025-07-15 16:00", tz="Asia/Hong_Kong").tz_convert("America/New_York").hour < 9)

Stocks: daily log return of each stock, shifted by 1. We keep only the stocks with no missing values in the period. They are kept for a PCA that we will do later, fitted on train only.

In [157]:
not_stocks = tickers1 + tickers2 + ["^NDX"]
stocks = [t for t in data["Close"].columns if t not in not_stocks]
stock_close = data["Close"][stocks].dropna(axis=1)
print(len(stocks), "stocks,", stock_close.shape[1], "with no missing values")

91 stocks, 81 with no missing values


In [158]:
stock_ret = np.log(stock_close).diff().shift(1)
stock_ret.columns = pd.MultiIndex.from_product([["stock_ret"], stock_ret.columns])

Breadth: % of those stocks with a positive return the day before.

In [159]:
data[("pct_up", "")] = ((stock_ret > 0).mean(axis=1) * 100).where(stock_ret.notna().all(axis=1))

## Join

The Open, Close and Volume levels are not features (they are replaced by the returns above), so we drop them. Then we do an inner join of the daily table and the NQ features on the `^NDX` trading dates.

In [160]:
data = data.drop(columns=["Open", "Close", "Volume"], level=0)
data = pd.concat([data, stock_ret], axis=1)

The first rows have no 20-day values (warm-up), so they have missing values. We remove the rows with missing values.

In [161]:
nulls = data.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

                 Ticker
ndx_ret_20d                21
ndx_vol_20d                21
ndx_ret_5d                  6
oil_ret                     4
nikkei_ret                  3
                           ..
yield_3m                    1
yield_10y                   1
vxn                         1
vix                         1
ndx_ret_ny_prev             1
Length: 96, dtype: int64


In [162]:
n_before = len(data)
data = data.dropna()
print("rows dropped:", n_before - len(data), "| first day:", data.index.min().date())

rows dropped: 23 | first day: 2018-02-01


In [163]:
# anti-leakage: no price level of day t in the features
feature_cols = [c for c in data.columns if c not in target_cols]
assert ("Close", "^NDX") not in feature_cols and ("Open", "^NDX") not in feature_cols
assert not data.columns.get_level_values(0).isin(["Open", "Close", "Volume"]).any()

In [164]:
print(data.shape)
print(data.index.min(), data.index.max())

(1975, 99)
2018-02-01 00:00:00 2025-12-11 00:00:00


In [165]:
data

Dayofweek    ret_ny y_ny ndx_ret_1d ndx_ret_5d ndx_ret_20d  \
Ticker                                                                  
Date                                                                    
2018-02-01         3 -0.001313  0.0   0.002775   0.004418    0.065195   
2018-02-02         4 -0.015571  0.0  -0.007001  -0.002142    0.048343   
2018-02-05         0 -0.029061  0.0  -0.020673  -0.038120    0.026335   
2018-02-06         1  0.044068  1.0  -0.039892  -0.073066   -0.023937   
2018-02-07         2 -0.007712  0.0   0.025843  -0.038948   -0.001596   
...              ...       ...  ...        ...        ...         ...   
2025-12-05         4  0.001335  1.0  -0.000971   0.013568   -0.001497   
2025-12-08         0 -0.005094  0.0   0.004304   0.010060    0.022118   
2025-12-09         1  0.002927  1.0  -0.002498   0.011187    0.022418   
2025-12-10         2  0.005635  1.0   0.001588   0.004405    0.002221   
2025-12-11         3  0.003443  1.0   0.004189   0.006613    0.009470   

           ndx_ret_ny_prev ndx_vol_20d        vix        vxn  ... stock_ret  \
Ticker                                                        ...      TMUS   
Date                                                          ...             
2018-02-01       -0.003278    0.006509  13.540000  19.650000  ...  0.005236   
2018-02-02       -0.001313    0.006699  13.470000  19.150000  ... -0.020328   
2018-02-05       -0.015571    0.008462  17.309999  20.660000  ... -0.003140   
2018-02-06       -0.029061    0.012247  37.320000  33.020000  ... -0.032769   
2018-02-07        0.044068    0.013638  29.980000  26.700001  ... -0.011276   
...                    ...         ...        ...        ...  ...       ...   
2025-12-05       -0.002984    0.012785  15.780000  20.190001  ... -0.002401   
2025-12-08        0.001335    0.011981  15.410000  19.620001  ...  0.007758   
2025-12-09       -0.005094    0.011976  16.660000  20.750000  ... -0.025070   
2025-12-10        0.002927    0.010949  16.930000  20.760000  ... -0.015230   
2025-12-11        0.005635    0.010958  15.770000  19.670000  ... -0.030405   

                                                                        \
Ticker           TRI      TSLA      TTWO       TXN      VRTX      WDAY   
Date                                                                     
2018-02-01 -0.071499  0.024254  0.068277 -0.002004  0.001980 -0.003081   
2018-02-02 -0.002543 -0.014384 -0.016156  0.003731  0.036019 -0.007199   
2018-02-05 -0.020108 -0.015873 -0.024117 -0.022229 -0.039801 -0.013022   
2018-02-06 -0.026809 -0.031382 -0.036322 -0.032570 -0.069105 -0.023597   
2018-02-07 -0.015895  0.002518  0.007980  0.015804  0.026273  0.011438   
...              ...       ...       ...       ...       ...       ...   
2025-12-05 -0.004627  0.017287  0.010397 -0.013675 -0.012537  0.008251   
2025-12-08 -0.001947  0.001034  0.001494  0.013346 -0.004119  0.015301   
2025-12-09 -0.028355 -0.034478 -0.002423 -0.008804 -0.029951  0.004717   
2025-12-10 -0.008051  0.012637  0.005847 -0.007879 -0.011444  0.005640   
2025-12-11  0.009284  0.014008 -0.010955  0.011905  0.017309  0.002247   

                                          
Ticker           WDC       WMT       XEL  
Date                                      
2018-02-01  0.021584 -0.010545  0.009466  
2018-02-02 -0.015060 -0.010183 -0.018353  
2018-02-05 -0.027764 -0.009905 -0.007392  
2018-02-06 -0.045599 -0.042926 -0.019295  
2018-02-07  0.033800  0.008060 -0.020844  
...              ...       ...       ...  
2025-12-05  0.034984  0.003751 -0.007941  
2025-12-08  0.047843  0.002348 -0.007615  
2025-12-09  0.005256 -0.013557 -0.018966  
2025-12-10 -0.001415  0.013122 -0.000132  
2025-12-11  0.070643 -0.016474 -0.014634  

[1975 rows x 99 columns]

In [166]:
data.to_feather("../data/final_data_ny_big.feather")

## Train / test split

Same as `Data_Splitting.ipynb`: chronological, the test set is the last 20% of the rows. Only used here for the checks.

In [167]:
split = int(0.8 * len(data))
train, test = data.iloc[:split], data.iloc[split:]

Class balance: see below. A model that always says "up" already gets that accuracy, that's the baseline to beat.

**Known limitation:** the overnight features are only daily closes of Asia, much less detailed than the NQ 1-minute bars of `1_Project_Features.ipynb`. The period also mixes very different markets (2008, 2020, 2022).

## Checks

In [168]:
print("final shape:", data.shape)
print("train:", train.shape, train.index.min().date(), "->", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "->", test.index.max().date())

final shape: (1975, 99)
train: (1580, 99) 2018-02-01 -> 2024-05-15
test:  (395, 99) 2024-05-16 -> 2025-12-11


In [169]:
print([c for c in feature_cols if c[0] != "stock_ret"])
print((data.columns.get_level_values(0) == "stock_ret").sum(), "stock_ret columns")

[('Dayofweek', ''), ('ndx_ret_1d', ''), ('ndx_ret_5d', ''), ('ndx_ret_20d', ''), ('ndx_ret_ny_prev', ''), ('ndx_vol_20d', ''), ('vix', ''), ('vxn', ''), ('yield_10y', ''), ('yield_3m', ''), ('dollar_ret', ''), ('oil_ret', ''), ('gold_ret', ''), ('nikkei_ret', ''), ('hsi_ret', ''), ('pct_up', '')]
81 stock_ret columns


In [170]:
print(train[("y_ny", "")].value_counts(normalize=True))
print(test[("y_ny", "")].value_counts(normalize=True))

(y_ny, )
1.0    0.537342
0.0    0.462658
Name: proportion, dtype: float64
(y_ny, )
1.0    0.559494
0.0    0.440506
Name: proportion, dtype: float64


Leakage sanity check: correlation of each feature with the target return on train (Asia features first, then the daily features). A value above 0.3 (in absolute value) would be suspicious.

In [171]:
corr = train[asia_cols].corrwith(train[("ret_ny", "")])
corr.index = [c[0] for c in corr.index]
print(corr.sort_values())
print("max |corr|:", corr.abs().max())

hsi_ret      -0.037916
nikkei_ret   -0.008105
dtype: float64
max |corr|: 0.03791582050072458


In [172]:
daily_cols = [c for c in feature_cols if c[0] != "stock_ret" and c not in asia_cols]
corr_daily = train[daily_cols].corrwith(train[("ret_ny", "")])
corr_daily.index = [c[0] for c in corr_daily.index]
print(corr_daily.sort_values())
print("max |corr|:", corr_daily.abs().max())

ndx_ret_1d        -0.119446
ndx_ret_ny_prev   -0.102843
pct_up            -0.054884
ndx_ret_5d        -0.052474
dollar_ret        -0.051707
oil_ret           -0.041795
ndx_ret_20d       -0.040159
Dayofweek         -0.014846
yield_10y         -0.010058
yield_3m           0.015420
gold_ret           0.021440
ndx_vol_20d        0.032162
vxn                0.049520
vix                0.059857
dtype: float64
max |corr|: 0.11944626937656769
